# Dropout Visualization

Each forward pass **randomly zeros** a fraction of neurons — different masks every time.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Model with dropout

In [ ]:
class DropNet(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.fc = nn.Linear(8, 16)
        self.drop = nn.Dropout(p=p)
    def forward(self, x):
        h = F.relu(self.fc(x))
        return self.drop(h)

model = DropNet(p=0.5)
x = torch.ones(1, 8)

## 2. Multiple forward passes — different masks

In [ ]:
masks = []
for _ in range(6):
    model.train()
    out = model(x).detach()
    mask = (out == 0).float().squeeze()
    masks.append(mask.numpy())
masks = np.array(masks)
print(f"mask shape: {masks.shape}")

## 3. Heatmap of dropped neurons per pass

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(masks, aspect='auto', cmap='Greys', vmin=0, vmax=1)
ax.set_xlabel('neuron index'); ax.set_ylabel('forward pass #')
ax.set_title('White = dropped neuron (Dropout p=0.5)')
plt.tight_layout(); plt.show()

## 4. Drop fraction per neuron across passes

In [ ]:
drop_rate = masks.mean(axis=0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(16), drop_rate, color='steelblue', edgecolor='black')
ax.axhline(0.5, color='red', ls='--', label='p=0.5 target')
ax.set_xlabel('neuron'); ax.legend()
ax.set_title('Empirical drop rate per neuron (6 passes)')
plt.tight_layout(); plt.show()

## 5. Train vs eval mode output scale

In [ ]:
model.train()
train_outs = torch.stack([model(x).detach() for _ in range(50)]).mean(0)
model.eval()
eval_out = model(x).detach()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(16), train_outs.squeeze().numpy(), color='coral')
axes[0].set_title('Mean output in train mode (noisy)')
axes[1].bar(range(16), eval_out.squeeze().numpy(), color='seagreen')
axes[1].set_title('Output in eval mode (deterministic)')
plt.tight_layout(); plt.show()